# SALCA Soil Quality (farm level)

## Live demonstration during Brightcon 2026

Mock data with example results displayed for **erosion and soil quality indicators of arable land**

Model based on [Oberholzer et al. (2012)](https://doi.org/10.1007/s13593-011-0072-7) and [Nemecek et al. (2024)](https://doi.org/10.1007/s11367-023-02255-w)

![](https://media.springernature.com/full/springer-static/image/art%3A10.1007%2Fs13593-011-0072-7/MediaObjects/13593_2011_72_Fig1_HTML.gif?as=webp)

Figure source: [Oberholzer et al. (2012)](https://doi.org/10.1007/s13593-011-0072-7)

![](SALCAsoilquality_steps_farm.png)

In [ ]:
"""
File name: SALCAsoilquality_farm.py
Created on Mon Oct 14 11:28:22 2019

Author: Agroscope, 'Life Cycle Assessment' Research Group, Switzerland
Date: 2026-09-16
Version: 1.0.1-demo
Licence: GNU LGPLv3

Description: 
    This script assesses the impact of agricultural management practices on
    soil quality.

Calculation level:
    Farm (entity comprising all fields, crops and animal groups)

Related scripts: SALCAerosion_field.py, SALCAheavymetal_crop.py,
                 SALCAmapping_ppp.py, SALCAanimal_farm.py, SALCAsoilquality_crop.py
    It uses input from these scripts.
"""

In [ ]:
# %% import modules ===========================================================

import json as js
import warnings as w
from sys import stderr
from dataclasses import dataclass

import pandas as pd
import numpy as np

# from salcaconnector import create_salca_context  # only relevant to Agroscope-internal use

In [ ]:
# %% start sequence ===========================================================

class SALCAconnectorInput:
    def __init__(self, d = None):
        if d is not None:
            for key, value in d.items():
                setattr(self, key, value)

class SALCAconnectorOutput:
    pass

with open('SALCAsoilquality_farm_data.json', mode = 'r') as f:
    inp = SALCAconnectorInput(js.load(f))

out = SALCAconnectorOutput()

In [ ]:
# %% define classes ===========================================================

@dataclass
class InputGeneral:
    """
    Data class collecting the input data needed in this module
    beyond the output of SALCAsoiluqality at the crop level.
    """

    c_areatime: list[float]
    fi_eros_mmpa: np.ndarray
    fi_fia_ha: np.ndarray

@dataclass
class LandUseConfig:
    """Data class defining the configuration of arable or grassland."""
    
    name: str
    name_alt: str
    weights: np.ndarray
    area_time: float

In [ ]:
# %% define auxiliary functions ===============================================

def weighted_average(values, cfg):
    """Calculate an (area-)weighted average."""
    return np.sum(np.multiply(values, cfg.weights))


def weighted_average_nan(values, cfg):
    """
    Calculate a weighted average without considering NaNs.
    Assumption that average conditions also apply to missing cases.
    np.nan if all values are missing.
    """
    f_value = np.ma.average(
        np.ma.masked_array(values,
                           mask = [1 if np.isnan(x) else 0 for x in values]),
        weights = cfg.weights)
    if f_value is np.ma.masked:
        f_value = np.nan
    return f_value


def compare_against_thresholds(value, thresholds):
    """Compare soil quality indicator value against indicator-specific thresholds."""
    if value > thresholds[0]:
        return 2
    elif value > thresholds[1]:
        return 1
    elif value >= thresholds[2]:
        return 0
    elif value > thresholds[3]:
        return -1
    else:
        return -2

In [ ]:
# %% define functions for impact classes ======================================

def classify_erosion(value):
    """Classify risk of soil erosion."""
    if value < 1.2:
        return 0
    elif value <= 9.6:
        return -1
    else:
        return -2


# wheeling classified at the crop level

def classify_grazing(value):
    """
    Classify risk of soil compaction by grazing.
    The actual classification is conducted at the crop level.
    """
    if not np.isnan(value):
        return value

    w.warn("Grazing has not been evaluated.")
    stderr.flush()  # show warning immediately
    return 0


def classify_structure_stabilisation(value):
    """Classify stabilisation of soil structure by plants."""
    if value > 0.7:
        return 2
    elif value > 0.4:
        return 1
    else:
        return 0


def classify_structure_formation(value):
    """Classify structure build-up by straw amendment."""
    if value > 0.3:
        return 1
    else:
        return 0


def classify_humus_balance(value):
    """Classify humus dynamics."""
    if value < -400:
        return -2
    elif value < -200:
        return -1
    elif value <= 200:
        return 0
    else:
        return 1


def classify_earthworm_protection_plants(value):
    """Classify positive effects of plants on earthworm populations."""
    if value > 0.7:
        return 1
    else:
        return 0


def classify_earthworm_protection_cutting(value):
    """Classify positive effects of high cutting levels on earthworm populations."""
    if value > 0.5:
        return 1
    else:
        return 0


def classify_earthworm_impact_tillage(value):
    """Classify potential impact of soil tillage on earthworms."""
    if value > 1:
        return -1
    else:
        return 0


def classify_earthworm_impact_seedbed(value):
    """Classify potential impact of seedbed preparation on earthworms."""
    if value > 0.66:
        return 1
    else:
        return 0


def classify_pollutants(values):
    """Classify input of heavy metals and organic pollutants."""
    # omit missing values
    valid_values = [x for x in values if not np.isnan(x)]

    # consider only the shortest time
    if not valid_values:  # empty list
        min_value = np.nan
    else:
        min_value = min(valid_values)

    # categorize
    if min_value > 300:
        return 0
    elif min_value >= 30:
        return -1
    elif min_value < 30:
        return -2
    else:
        return np.nan


def classify_slurry_application(value):
    """Classify toxic effects of slurry application."""
    if value > 0.5:
        return -1
    else:
        return 0


def classify_stable_organic_substances(value):
    """Classify input of stable organic fertiliser."""
    if value > 1500:
        return 1
    else:
        return 0


def classify_degrad_organic_substances(value):
    """Classify input of rapidly degradable organic fertiliser."""
    if value > 1800:
        return 1
    else:
        return 0


def classify_liming(value):
    """Classify liming at pH < 6.2."""
    if value >= 0.8:
        return 1
    else:
        return 0


def classify_ppp_application(value):
    """Classify toxic effects of pesticide application."""
    if value > 0.33:
        return -1
    else:
        return 0

In [ ]:
# %% define functions for indicators (soil properties) ========================

# aggregation of the impact classes contributing to each of the nine indicators

def evaluate_rooting_depth(erosion):
    """
    Evaluate rooting depth.
    This is one of three physical soil quality indicators.
    """
    weight_erosion = 1
    return erosion * weight_erosion


def evaluate_pores_aggregates(*, land_use, treatments, grazing = None, st_stab,
                                 st_form = None, humus = None, org_stable, pH):
    """
    Evaluate macropore volume and aggregate stability.
    These are two of three physical soil quality indicators.
    The only difference of aggregate stability to macropore volume is the
    additional consideration of humus, but only for arable areas.
    """
    weight_treatment = 1
    weight_grazing = 0 if land_use == 'arable' else 0.5
    weight_st_stab = 1 if land_use == 'arable' else 0.5
    weight_st_form = 1/3 if land_use == 'arable' else 0
    weight_ph = 1/3 if land_use == 'arable' else 0.5
    weight_org_stable = 1/3 if land_use == 'arable' else 0.5
    weight_humus = 2 if land_use == 'arable' else 0

    if grazing is None:
        grazing = 0
    if st_form is None:
        st_form = 0
    if humus is None:  # if grassland or evaluation of macropore volume
        humus = 0

    disturbance = (treatments * weight_treatment
                   + grazing * weight_grazing)

    improvement = (st_stab * weight_st_stab
                   + st_form * weight_st_form
                   + pH * weight_ph
                   + org_stable * weight_org_stable
                   + humus * weight_humus)

    return improvement + disturbance


def evaluate_organic_carbon(humus):
    """
    Evaluate organic carbon.
    This is one of three chemical soil quality indicators.
    """
    weight_humus = 1
    return humus * weight_humus


def evaluate_heavy_metals(hm):
    """
    Evaluate heavy metals.
    This is one of three chemical soil quality indicators.
    """
    weight_hm = 1
    return hm * weight_hm


def evaluate_organic_pollutants(op):
    """
    Evaluate organic pollutants.
    This is one of three chemical soil quality indicators.
    """
    weight_op = 1
    return op * weight_op


def evaluate_earthworm_biomass(*, land_use, ew_slurry, ew_ppp, st_form = None,
                               ew_cult, ew_cutting = None, ew_phys_dist = None,
                               ew_seedbed = None, pH, org_stable):
    """
    Evaluate earthworm biomass.
    This is one of three biological soil quality indicators.
    """
    weight_ew_slurry = 1
    weight_ew_ppp = 2
    weight_st_form = 0.5 if land_use == 'arable' else 0
    weight_ew_cult = 1
    weight_ew_cutting = 0 if land_use == 'arable' else 0.5
    weight_ew_phys_dist = 2 if land_use == 'arable' else 0
    weight_ew_seedbed = 2 if land_use == 'arable' else 0
    weight_ph = 1
    weight_org_stable = 0.5

    if st_form is None:
        st_form = 0
    if ew_cutting is None:
        ew_cutting = 0
    if ew_phys_dist is None:
        ew_phys_dist = 0
    if ew_seedbed is None:
        ew_seedbed = 0

    disturbance = (ew_phys_dist * weight_ew_phys_dist
                   + ew_slurry * weight_ew_slurry
                   + ew_ppp * weight_ew_ppp)

    improvement = (ew_cult * weight_ew_cult
                   + ew_seedbed * weight_ew_seedbed
                   + ew_cutting * weight_ew_cutting
                   + pH * weight_ph
                   + org_stable * weight_org_stable
                   + st_form * weight_st_form)

    return improvement + disturbance


def evaluate_microbes(*, land_use, mp_ppp, lp_balance, hm_balance, op_balance,
                      humus = None, org_stable, org_degrad = None, pH):
    """
    Evaluate microbial biomass and microbial activity.
    These are two of three biological soil quality indicators.
    The only difference of microbial activity to microbial biomass is the
    additional consideration of degradable organic substances.
    """
    weight_mb_ppp = 0.5
    weight_lp = 1
    weight_hm = 1
    weight_op = 1
    weight_humus = 1 if land_use == 'arable' else 0
    weight_org_stable = 1
    weight_org_degrad = 0.5
    weight_ph = 1

    if humus is None:
        humus = 0
    if org_degrad is None:  # if evaluation of microbial biomass
        org_degrad = 0

    if lp_balance <= -3:
        im_lp = -2
    elif lp_balance < -1:
        im_lp = -1
    else:
        im_lp = 0

    if not np.isnan(op_balance):
        disturbance = (mp_ppp * weight_mb_ppp
                       + im_lp * weight_lp
                       + hm_balance * weight_hm
                       + op_balance * weight_op)
    else:
        disturbance = (mp_ppp * weight_mb_ppp
                       + im_lp * weight_lp
                       + hm_balance * weight_hm)

    improvement = (pH * weight_ph
                   + org_stable * weight_org_stable
                   + org_degrad * weight_org_degrad
                   + humus * weight_humus)

    return improvement + disturbance


def classify_indicators(summary_dict):
    """Classify soil quality indicators into scores from -2 to 2."""
    ev_result = {}

    # rooting depth
    ev_result['rooting depth'] = summary_dict['rooting depth']

    # large pores
    ev_result['large pores'] = \
        compare_against_thresholds(summary_dict['large pores'], [2, 1, -1, -3])

    # soil texture stability
    ev_result['soil texture stability'] = \
        compare_against_thresholds(summary_dict['soil texture stability'], [3, 1, -1.5, -5])

    # Corg content: only for arable, not for grass
    ev_result['Corg content'] = summary_dict['Corg content']

    # organic pollutants
    ev_result['organic pollutants'] = summary_dict['organic pollutants']

    # heavy metal content
    ev_result['heavy metal content'] = summary_dict['heavy metal content']

    # earthworm biomass
    ev_result['earthworm biomass'] = \
        compare_against_thresholds(summary_dict['earthworm biomass'], [3, 1, -1, -3])

    # microbial biomass
    ev_result['microbial biomass'] = \
        compare_against_thresholds(summary_dict['microbial biomass'], [2, 1, -1, -3])

    # microbial activity
    ev_result['microbial activity'] = \
        compare_against_thresholds(summary_dict['microbial activity'], [2, 1, -1, -3])

    return ev_result

In [ ]:
# %% load input data ==========================================================

inp_general = InputGeneral(
    c_areatime = [js.loads(i)['c_areatime'] for i in inp.c_salcaprep],  # area-time (crop)
    fi_eros_mmpa = np.array(inp.fi_eros_mmpa),  # erosion [kg/ha/a] from SALCAerosion_field
    fi_fia_ha = np.array(inp.fi_fia_ha)  # field area [ha]
)

# input from SALCAsoilquality_crop
sq_crop = [js.loads(var) for var in inp.c_sq_output]

# variables introduced only recently and not included in all projects
keys_new = ['c_graz_evaluation',
            'c_ttt_arable_pah_a',
            'c_ttt_arable_pcb_a',
            'c_ttt_arable_pcdd_pcdf_a',
            'c_ttt_grass_pah_a',
            'c_ttt_grass_pcb_a',
            'c_ttt_grass_pcdd_pcdf_a']

# convert a list of dicts to a dict of np.arrays
sq_crop = {
    key: np.array([crop.get(key, np.nan) for crop in sq_crop])
    for key in sorted(list(set(list(sq_crop[0].keys()) + keys_new)))
}

In [ ]:
# %% aggregate crop values on farm level ======================================

# sum of area-time for arable and grassland areas
f_areatime_arable = \
    sum(a for a, b in zip(inp_general.c_areatime, sq_crop['c_is_arable']) if b)
f_areatime_grass = \
    sum(a for a, b in zip(inp_general.c_areatime, sq_crop['c_is_grass']) if b)

# set up configurations for easier looping
arable_cfg = LandUseConfig(
    name = 'arable',
    name_alt = 'arable',
    weights = np.multiply(np.divide(inp_general.c_areatime, f_areatime_arable,
                                    where = f_areatime_arable != 0),
                          sq_crop['c_is_arable']),
    area_time = f_areatime_arable
)

grass_cfg = LandUseConfig(
    name = 'grass',
    name_alt = 'grassland',
    weights = np.multiply(np.divide(inp_general.c_areatime, f_areatime_grass,
                                    where = f_areatime_grass != 0),
                          sq_crop['c_is_grass']),
    area_time = f_areatime_grass
)

del f_areatime_arable, f_areatime_grass  # integrated into LandUseConfig

# aggregate
sq_farm = {}

for cfg in [globals()[var] for var in ['arable_cfg', 'grass_cfg']]:
    if cfg.area_time != 0:
        sq_farm[cfg.name] = {
            'tr_evaluation': weighted_average(sq_crop['c_tr_evaluation'], cfg),
            'structure_stabilisation':
                weighted_average(sq_crop['c_crop_rotation_' + cfg.name], cfg),
            'ttt_cd_a': weighted_average(sq_crop['c_ttt_' + cfg.name + '_cd_a'], cfg),
            'ttt_cu_a': weighted_average(sq_crop['c_ttt_' + cfg.name + '_cu_a'], cfg),
            'ttt_zn_a': weighted_average(sq_crop['c_ttt_' + cfg.name + '_zn_a'], cfg),
            'ttt_pb_a': weighted_average(sq_crop['c_ttt_' + cfg.name + '_pb_a'], cfg),
            'ttt_ni_a': weighted_average(sq_crop['c_ttt_' + cfg.name + '_ni_a'], cfg),
            'ttt_cr_a': weighted_average(sq_crop['c_ttt_' + cfg.name + '_cr_a'], cfg),
            'ttt_hg_a': weighted_average(sq_crop['c_ttt_' + cfg.name + '_hg_a'], cfg),
            'ttt_pcdd_pcdf_a':
                weighted_average_nan(sq_crop['c_ttt_' + cfg.name + '_pcdd_pcdf_a'], cfg),
            'ttt_pah_a': weighted_average_nan(sq_crop['c_ttt_' + cfg.name + '_pah_a'], cfg),
            'ttt_pcb_a': weighted_average_nan(sq_crop['c_ttt_' + cfg.name + '_pcb_a'], cfg),
            'ew_chem': weighted_average(sq_crop['c_evaluation_slurry_' + cfg.name_alt], cfg),
            'org_subst_stable_kgpha': weighted_average(sq_crop['c_org_subst_stable_kgpha'], cfg),
            'org_subst_degr_kgpha': weighted_average(sq_crop['c_org_subst_degradable_kgpha'], cfg),
            'pH_lime': weighted_average(sq_crop['c_pH_lime_' + cfg.name], cfg),
            'ew_chem_ppp': weighted_average(sq_crop['c_evaluation_ppp_' + cfg.name_alt], cfg),
            'mb_chem_ppp': weighted_average(sq_crop['c_evaluation_ppp_' + cfg.name_alt], cfg)
            }

if arable_cfg.area_time != 0:
    sq_farm['arable']['erosion_mmpa'] = \
        np.sum(np.multiply(inp_general.fi_eros_mmpa,
                           inp_general.fi_fia_ha / np.sum(inp_general.fi_fia_ha)))
    sq_farm['arable']['structure_formation'] = \
        weighted_average(sq_crop['c_soil_texture_hres_frac'], arable_cfg)
    sq_farm['arable']['humus_balance_kgpha'] = np.nansum(np.multiply(
        sq_crop['c_humus_balance_gross_kgpha'], arable_cfg.weights))
    sq_farm['arable']['humus_balance_area_ha'] = np.nansum(sq_crop['c_humus_balance_area_ha'])
    sq_farm['arable']['ew_tillage'] = weighted_average(sq_crop['c_ew_phys_sum'], arable_cfg)
    sq_farm['arable']['ew_sowi'] = weighted_average(sq_crop['c_protective_sowi'], arable_cfg)

if grass_cfg.area_time != 0:
    sq_farm['grass']['graz_evaluation'] = weighted_average(sq_crop['c_graz_evaluation'], grass_cfg)
    sq_farm['grass']['ew_cutting'] = \
        weighted_average(sq_crop['c_worm_protection_cutting_level'], grass_cfg)

In [ ]:
# %% classify impact classes ==================================================

ic_score = {}

if arable_cfg.area_time != 0:  # assessment only if the farm has arable area
    ic_score['arable'] = {
        'erosion': classify_erosion(sq_farm['arable']['erosion_mmpa']),
        'wheeling': sq_farm['arable']['tr_evaluation'],
        'grazing': 0,  # evaluation of grazing not relevant to arable land
        'st_stab': classify_structure_stabilisation(sq_farm['arable']['structure_stabilisation']),
        'st_form': classify_structure_formation(sq_farm['arable']['structure_formation']),
        'humus': classify_humus_balance(sq_farm['arable']['humus_balance_kgpha']),
        'ew_cult':
            classify_earthworm_protection_plants(sq_farm['arable']['structure_stabilisation']),
        'ew_phys_dist': classify_earthworm_impact_tillage(sq_farm['arable']['ew_tillage']),
        'ew_seedbed': classify_earthworm_impact_seedbed(sq_farm['arable']['ew_sowi']),
        'hm': classify_pollutants([sq_farm['arable']['ttt_cd_a'],
                                 sq_farm['arable']['ttt_cu_a'],
                                 sq_farm['arable']['ttt_zn_a'],
                                 sq_farm['arable']['ttt_pb_a'],
                                 sq_farm['arable']['ttt_ni_a'],
                                 sq_farm['arable']['ttt_cr_a'],
                                 sq_farm['arable']['ttt_hg_a']]),
        'op': classify_pollutants([sq_farm['arable']['ttt_pcdd_pcdf_a'],
                                 sq_farm['arable']['ttt_pah_a'],
                                 sq_farm['arable']['ttt_pcb_a']]),
        'ew_slurry': classify_slurry_application(sq_farm['arable']['ew_chem']),
        'org_stable':
            classify_stable_organic_substances(sq_farm['arable']['org_subst_stable_kgpha']),
        'org_degrad': classify_degrad_organic_substances(sq_farm['arable']['org_subst_degr_kgpha']),
        'pH': classify_liming(sq_farm['arable']['pH_lime']),
        'ew_ppp': classify_ppp_application(sq_farm['arable']['ew_chem_ppp']),
        'mp_ppp': classify_ppp_application(sq_farm['arable']['mb_chem_ppp'])
        }

if grass_cfg.area_time != 0:  # assessment only if the farm has grassland
    ic_score['grass'] = {
        'erosion': 0,  # evaluation of erosion not relevant to grassland
        'wheeling': sq_farm['grass']['tr_evaluation'],
        'grazing': classify_grazing(sq_farm['grass']['graz_evaluation']),
        'st_stab': classify_structure_stabilisation(sq_farm['grass']['structure_stabilisation']),
        'ew_cult':
            classify_earthworm_protection_plants(sq_farm['grass']['structure_stabilisation']),
        'ew_cutting': classify_earthworm_protection_cutting(sq_farm['grass']['ew_cutting']),
        'hm': classify_pollutants([sq_farm['grass']['ttt_cd_a'],
                                   sq_farm['grass']['ttt_cu_a'],
                                   sq_farm['grass']['ttt_zn_a'],
                                   sq_farm['grass']['ttt_pb_a'],
                                   sq_farm['grass']['ttt_ni_a'],
                                   sq_farm['grass']['ttt_cr_a'],
                                   sq_farm['grass']['ttt_hg_a']]),
        'op': classify_pollutants([sq_farm['grass']['ttt_pcdd_pcdf_a'],
                                   sq_farm['grass']['ttt_pah_a'],
                                   sq_farm['grass']['ttt_pcb_a']]),
        'ew_slurry': classify_slurry_application(sq_farm['grass']['ew_chem']),
        'org_stable':
            classify_stable_organic_substances(sq_farm['grass']['org_subst_stable_kgpha']),
        'org_degrad': classify_degrad_organic_substances(sq_farm['grass']['org_subst_degr_kgpha']),
        'pH': classify_liming(sq_farm['grass']['pH_lime']),
        'ew_ppp': classify_ppp_application(sq_farm['grass']['ew_chem_ppp']),
        'mp_ppp': classify_ppp_application(sq_farm['grass']['mb_chem_ppp'])
        }

In [ ]:
# %% evaluate indicators (soil properties) ====================================

sqi = {}

if arable_cfg.area_time != 0: #  assessment only if the farm has arable area
    # physical
    ev_root_depth_arable = evaluate_rooting_depth(ic_score['arable']['erosion'])
    lp_balance_arable = evaluate_pores_aggregates(land_use = 'arable',
        treatments = ic_score['arable']['wheeling'], st_stab = ic_score['arable']['st_stab'],
        st_form = ic_score['arable']['st_form'],
        org_stable = ic_score['arable']['org_stable'], pH = ic_score['arable']['pH'])
    sts_balance_arable = evaluate_pores_aggregates(land_use = 'arable',
        treatments = ic_score['arable']['wheeling'], st_stab = ic_score['arable']['st_stab'],
        st_form = ic_score['arable']['st_form'], humus = ic_score['arable']['humus'],
        org_stable = ic_score['arable']['org_stable'], pH = ic_score['arable']['pH'])

    # chemical
    coc_balance_arable = evaluate_organic_carbon(ic_score['arable']['humus'])
    hm_balance_arable = evaluate_heavy_metals(ic_score['arable']['hm'])
    op_balance_arable = evaluate_organic_pollutants(ic_score['arable']['op'])

    # biological
    ewb_balance_arable = evaluate_earthworm_biomass(land_use = 'arable',
        ew_slurry = ic_score['arable']['ew_slurry'], ew_ppp = ic_score['arable']['ew_ppp'],
        st_form = ic_score['arable']['st_form'], ew_cult = ic_score['arable']['ew_cult'],
        ew_phys_dist = ic_score['arable']['ew_phys_dist'],
        ew_seedbed = ic_score['arable']['ew_seedbed'],
        pH = ic_score['arable']['pH'], org_stable = ic_score['arable']['org_stable'])
    mb_balance_arable = evaluate_microbes(land_use = 'arable',
        mp_ppp = ic_score['arable']['mp_ppp'], lp_balance = lp_balance_arable,
        hm_balance = hm_balance_arable, op_balance = op_balance_arable,
        humus = ic_score['arable']['humus'], org_stable = ic_score['arable']['org_stable'],
        pH = ic_score['arable']['pH'])
    ma_balance_arable = evaluate_microbes(land_use = 'arable',
        mp_ppp = ic_score['arable']['mp_ppp'], lp_balance = lp_balance_arable,
        hm_balance = hm_balance_arable, op_balance = op_balance_arable,
        humus = ic_score['arable']['humus'], org_stable = ic_score['arable']['org_stable'],
        org_degrad = ic_score['arable']['org_degrad'], pH = ic_score['arable']['pH'])

    # summary
    sqi['arable'] = {'rooting depth': ev_root_depth_arable,
                     'large pores': lp_balance_arable,
                     'soil texture stability': sts_balance_arable,
                     'Corg content': coc_balance_arable,
                     'heavy metal content': hm_balance_arable,
                     'organic pollutants': op_balance_arable,
                     'earthworm biomass': ewb_balance_arable,
                     'microbial biomass': mb_balance_arable,
                     'microbial activity': ma_balance_arable
                     }

if grass_cfg.area_time != 0:  # assessment only if the farm has grassland
    # physical
    ev_root_depth_grass = 'nan'  # rooting depth only evaluated for arable areas
    lp_balance_grass = evaluate_pores_aggregates(land_use = 'grass',
        treatments = ic_score['grass']['wheeling'], grazing = ic_score['grass']['grazing'],
        st_stab = ic_score['grass']['st_stab'], org_stable = ic_score['grass']['org_stable'],
        pH = ic_score['grass']['pH'])
    sts_balance_grass = lp_balance_grass  # they only differ for arable areas

    # chemical
    coc_balance_grass = 'nan'  # organic carbon only evaluated for arable areas
    hm_balance_grass = evaluate_heavy_metals(ic_score['grass']['hm'])
    op_balance_grass = evaluate_organic_pollutants(ic_score['grass']['op'])

    # biological
    ewb_balance_grass = evaluate_earthworm_biomass(land_use = 'grass',
        ew_slurry = ic_score['grass']['ew_slurry'], ew_ppp = ic_score['grass']['ew_ppp'],
        ew_cult = ic_score['grass']['ew_cult'], ew_cutting = ic_score['grass']['ew_cutting'],
        pH = ic_score['grass']['pH'], org_stable = ic_score['grass']['org_stable'])
    mb_balance_grass = evaluate_microbes(land_use = 'grass',
        mp_ppp = ic_score['grass']['mp_ppp'], lp_balance = lp_balance_grass,
        hm_balance = hm_balance_grass, op_balance = op_balance_grass,
        org_stable = ic_score['grass']['org_stable'], pH = ic_score['grass']['pH'])
    ma_balance_grass = evaluate_microbes(land_use = 'grass',
        mp_ppp = ic_score['grass']['mp_ppp'], lp_balance = lp_balance_grass,
        hm_balance = hm_balance_grass, op_balance = op_balance_grass,
        org_stable = ic_score['grass']['org_stable'], org_degrad = ic_score['grass']['org_degrad'],
        pH = ic_score['grass']['pH'])

    # summary
    sqi['grass'] = {'rooting depth': ev_root_depth_grass,
                    'large pores': lp_balance_grass,
                    'soil texture stability': sts_balance_grass,
                    'Corg content': coc_balance_grass,
                    'heavy metal content': hm_balance_grass,
                    'organic pollutants': op_balance_grass,
                    'earthworm biomass': ewb_balance_grass,
                    'microbial biomass': mb_balance_grass,
                    'microbial activity': ma_balance_grass
                    }

In [ ]:
# %% classify indicators (soil properties) ====================================

sqi_score = {}

for cfg in [globals()[var] for var in ['arable_cfg', 'grass_cfg']]:
    if cfg.area_time != 0:
        sqi_score[cfg.name] = classify_indicators(sqi[cfg.name])
    else:
        sqi_score[cfg.name] = {'rooting depth': 'nan',
                               'large pores': 'nan',
                               'soil texture stability': 'nan',
                               'Corg content': 'nan',
                               'heavy metal content': 'nan',
                               'organic pollutants': 'nan',
                               'earthworm biomass': 'nan',
                               'microbial biomass': 'nan',
                               'microbial activity': 'nan'}

In [ ]:
# %% export output ============================================================

out.f_sq_arable_tier1 = js.dumps(sqi_score['arable'])
out.f_sq_grass_tier1 = js.dumps(sqi_score['grass'])

In [ ]:
# %% example results for soil quality indicators ==============================

print('Evaluation of erosion (mm/a):', sq_farm['arable']['erosion_mmpa'], '\n')

colours = {"-2": '\033[41m', "-1": '\033[41m', "0": '\033[30;43m',
           "1": '\033[42m', "2": '\033[42m', "nan": '\033[30;47m',
           "endc": '\033[0m'}

count = 0
print('\033[1m\033[4mClassification of soil quality indicators\033[0m')
for key, value in sqi_score['arable'].items():
    if count % 3 == 0 and count != 0:
        print('---')
    print(f"{key}: {colours[str(value)]}{value}{colours['endc']}")
    count = count + 1